### 01. First test: Capturing and Inspecting Raw Telemetry

This initial cell acts as a sanity check. It connects to the `raw.vehicle-positions` Kafka topic as a temporary consumer and pulls a finite batch of 500 real messages. 

The goal here is not to compute analytics yet, but to observe the raw shape, noise, and cadence of the MBTA data. I extracted the GeoJSON `location` coordinates into flat `lat` and `lon` columns, loaded the batch into a `pandas` DataFrame, and used DuckDB to sort the results chronologically per vehicle. This gives us our first raw glimpse into a vehicle's breadcrumb trail.

In [23]:
import json
import time
import pandas as pd
import duckdb
from confluent_kafka import Consumer, KafkaException

conf = {
    'bootstrap.servers': 'localhost:9092',
    'group.id': f'duckdb-explorer-{int(time.time())}',
    'auto.offset.reset': 'earliest',
    'enable.auto.commit': False,
}

consumer = Consumer(conf)
topic = 'raw.vehicle-positions'
consumer.subscribe([topic])

print(f"Collecting data from topic '{topic}'...")

messages = []
TARGET = 500
MAX_WAIT_SECONDS = 30
start = time.time()

try:
    while len(messages) < TARGET and (time.time() - start) < MAX_WAIT_SECONDS:
        msg = consumer.poll(timeout=1.0)
        if msg is None:
            continue
        if msg.error():
            raise KafkaException(msg.error())

        payload = json.loads(msg.value().decode('utf-8'))

        if 'location' in payload and 'coordinates' in payload['location']:
            payload['lon'] = payload['location']['coordinates'][0]
            payload['lat'] = payload['location']['coordinates'][1]

        messages.append(payload)

except KeyboardInterrupt:
    print("Capture manually stoped")
finally:
    consumer.close()

df_analitics = pd.DataFrame(messages)
print(f"\n{len(df_pings)} pings stored in Pandas")

query = """
    SELECT 
        vehicle_id,
        trip_id,
        route_id,
        timestamp,
        lat,
        lon
    FROM df_analitics
    ORDER BY vehicle_id, timestamp
"""

df_analitics = duckdb.sql(query).df()
df_analitics.head(10)


1000 pings stored in Pandas


,vehicle_id,trip_id,route_id,timestamp,lat,lon
0,G-10033,ADDED-1584850870,Green-D,2026-08-15T05:36:22.000Z,42.377048,-71.093964
1,G-10033,ADDED-1584850870,Green-D,2026-08-15T05:36:22.000Z,42.377048,-71.093964
2,G-10033,ADDED-1584850870,Green-D,2026-08-15T05:36:22.000Z,42.377048,-71.093964
3,G-10033,ADDED-1584850870,Green-D,2026-08-15T05:37:23.000Z,42.377048,-71.093964
4,G-10033,ADDED-1584850870,Green-D,2026-08-15T05:37:23.000Z,42.377048,-71.093964
5,G-10033,ADDED-1584850870,Green-D,2026-08-15T05:37:23.000Z,42.377048,-71.093964
6,G-10033,ADDED-1584850870,Green-D,2026-08-15T05:37:23.000Z,42.377048,-71.093964
7,G-10033,ADDED-1584850870,Green-D,2026-08-15T05:38:24.000Z,42.377048,-71.093964
8,G-10033,ADDED-1584850870,Green-D,2026-08-15T06:04:50.000Z,42.377048,-71.093964
9,G-10033,ADDED-1584850870,Green-D,2026-08-15T06:04:50.000Z,42.377048,-71.093964


### 02. Second test: Deduplication and Real Update Frequency (Gaps)

After inspecting the initial raw batch, I noticed duplicate consecutive pings for the same vehicle (identical timestamps and coordinates). This happens because our ingestion service polls the MBTA feed (every 10-15 seconds) faster than the actual vehicles update their internal GPS.

To compute accurate metrics later on, we must filter out this noise. This cell uses DuckDB to deduplicate consecutive identical records. Then, it applies the `LAG()` window function to calculate the actual time gap (in seconds) between genuine vehicle movements, revealing the true update frequency of our telemetry stream.

In [30]:
query_gaps = """
    WITH deduplicated_pings AS (
        -- Deleting identical records
        SELECT 
            vehicle_id,
            trip_id,
            route_id,
            CAST(timestamp AS TIMESTAMPTZ) AS timestamp,
            lat,
            lon
        FROM df_analitics
        GROUP BY vehicle_id, trip_id, route_id, timestamp, lat, lon
    )
    -- Calculate the diff between pings for every vehicle
    SELECT 
        vehicle_id,
        timestamp,
        trip_id,
        route_id,
        lat,
        lon,
        LAG(timestamp) OVER (PARTITION BY vehicle_id ORDER BY timestamp) AS prev_timestamp,
        date_diff('second', 
            LAG(timestamp) OVER (PARTITION BY vehicle_id ORDER BY timestamp), 
            timestamp
        ) AS real_update_gap_seconds
    FROM deduplicated_pings
    ORDER BY vehicle_id, timestamp
"""

df_gaps = duckdb.sql(query_gaps).df()
display(df_gaps.head(15))

,vehicle_id,timestamp,trip_id,route_id,lat,lon,prev_timestamp,real_update_gap_seconds
0,G-10033,2026-08-14 23:36:22-06:00,ADDED-1584850870,Green-D,42.377048,-71.093964,NaT,<NA>
1,G-10033,2026-08-14 23:37:23-06:00,ADDED-1584850870,Green-D,42.377048,-71.093964,2026-08-14 23:36:22-06:00,61
2,G-10033,2026-08-14 23:38:24-06:00,ADDED-1584850870,Green-D,42.377048,-71.093964,2026-08-14 23:37:23-06:00,61
3,G-10033,2026-08-15 00:04:50-06:00,ADDED-1584850870,Green-D,42.377048,-71.093964,2026-08-14 23:38:24-06:00,1586
4,G-10033,2026-08-15 00:05:51-06:00,ADDED-1584850870,Green-D,42.377048,-71.093964,2026-08-15 00:04:50-06:00,61
5,G-10033,2026-08-15 00:06:52-06:00,ADDED-1584850870,Green-D,42.377048,-71.093964,2026-08-15 00:05:51-06:00,61
6,G-10033,2026-08-15 00:07:53-06:00,ADDED-1584850870,Green-D,42.377048,-71.093964,2026-08-15 00:06:52-06:00,61
7,G-10040,2026-08-14 23:35:43-06:00,ADDED-1584850600,Green-E,42.408401,-71.117493,NaT,<NA>
8,G-10040,2026-08-14 23:36:44-06:00,ADDED-1584850600,Green-E,42.408401,-71.117493,2026-08-14 23:35:43-06:00,61
9,G-10040,2026-08-14 23:37:45-06:00,ADDED-1584850600,Green-E,42.408401,-71.117493,2026-08-14 23:36:44-06:00,61


### 02.5. Gap Distribution & Silent Vehicle Thresholds

To build a reliable "stale vehicle" detection system, it must be established a baseline for normal update frequencies. This cell calculates the summary statistics and quantiles for `real_update_gap_seconds` metric. 

The data reveals that while the median update gap is 16 seconds, the 99th percentile sits at 76 seconds. This provides a data-driven threshold: if a vehicle goes silent for more than ~90-120 seconds, we can confidently flag it as a feed interruption, a "silent vehicle", or an out-of-service anomaly, rather than normal network jitter.

In [31]:
df_gaps['real_update_gap_seconds'].describe()

count         258.0
mean      150.46124
std      428.029737
min             5.0
25%            13.0
50%            16.0
75%            23.0
max          1888.0
Name: real_update_gap_seconds, dtype: Float64

In [32]:
df_gaps['real_update_gap_seconds'].quantile([0.5, 0.75, 0.9, 0.95, 0.99])

0.50       16.0
0.75       23.0
0.90      155.0
0.95     1597.3
0.99    1712.09
Name: real_update_gap_seconds, dtype: Float64

### 03. Third Attempt: Spatial Map Matching against GTFS-Static

Now that we have a clean, deduplicated sequence of vehicle pings, we need to anchor these raw GPS coordinates to the physical transit network. A raw lat/lon pair is useless if we don't know whether the vehicle is heading to stop 4 or driving parallel to stop 20 on its return trip.

This cell performs a spatial join between our real-time telemetry (`df_gaps`) and the MBTA's GTFS-static schedule (`stop_times.txt`, `stops.txt`, and `routes.txt`). By filtering strictly on the vehicle's current `trip_id` and calculating the squared Euclidean distance to all scheduled stops on that specific sequence, we can pinpoint the closest logical stop for every single ping. 

**Geospatial Correction & Metric Conversion:**

Originally, I was using the following calculation to get the distance between the vehicle and the next stop:

$((p.\text{lat} - s.\text{stop\_lat}) \times 111320)^2 + ((p.\text{lon} - s.\text{stop\_lon}) \times 111320 \times \cos(p.\text{lat}))^2$

But this had some problems: 

- First, the number retrieved had no real-world meaning: it as something like

$1.879e-05 \times 111320^2$

which was not useful because you cannot tell if that is 5 meters or 500 meters, which was the real answer I was looking for since I want to use it for bunching/proximity thresholds. 

- The second problem was the interpretation of the latitudes: a degree of longitude and a degree of latitude are not the same physical distance, except at the equator. At Boston's latitud, a degree of longitude is only about 74% as long as a degree of latitude, since latitude has a cos() compression in the earth. Treating them as equivalent means east-west differences get systematically under-weighted relative to north-south ones. TL;DR: The nearest stop was calculated wrong, and there were distorion tha was making the calculation ambiguous. 

To solve this, instead of computing raw squared Euclidean distance on degrees (which creates a distortion as mentioned, since 1 degree of longitude in Boston is only ~74% as long as 1 degree of latitude), I applied the Equirectangular approximation:

$$d^2 = (\Delta \text{lat} \times 111320)^2 + (\Delta \text{lon} \times 111320 \times \cos(\text{lat}))^2$$

By multiplying by 111,320 (the approximate meters in a degree) and apply the `COS(RADIANS(lat))` correction to the longitude. This gives us the squared distance in meters, allowing us to set meaningful bunching and proximity thresholds later on.

*Note: Real-world data is messy. During our initial run, DuckDB crashed with a `Conversion Error` because it expected `trip_id` to be a `BIGINT`. It turns out the MBTA created a custom alphanumeric trip ID (`8pmChrisBrownUsher-847359-4763`) for extra transit service during a Chris Brown and Usher concert. To prevent this from crashing the pipeline, we now explicitly force DuckDB to read all IDs as `VARCHAR` using the `types` parameter.*

In [43]:
# Next step: Spatial Map Matching with Routes Names
# Comparing data of the current position vs the strict sequence of the vehicle stops

query_map_matching = """
    WITH stops_for_trip AS (
        SELECT 
            p.vehicle_id,
            p.trip_id,
            p.timestamp,
            p.lat AS ping_lat,
            p.lon AS ping_lon,
            r.route_short_name,
            r.route_long_name,
            st.stop_sequence,
            s.stop_id,
            s.stop_name,
            
            -- Approximate meters-per-degree at Boston's latitude, corrected for longitude compression
            (
                POW((p.lat - s.stop_lat) * 111320, 2) +
                POW((p.lon - s.stop_lon) * 111320 * COS(RADIANS(p.lat)), 2)
            ) AS dist_sq_meters
        
        FROM df_gaps AS p
        -- Forcing DuckDB to read the IDs as VARCHAR to avoid failing at alphanumeric IDs
        JOIN read_csv_auto('../gtfs_static/MBTA_GTFS/stop_times.txt', types={'trip_id': 'VARCHAR', 'stop_id': 'VARCHAR'}) AS st 
            ON p.trip_id = st.trip_id
        JOIN read_csv_auto('../gtfs_static/MBTA_GTFS/stops.txt', types={'stop_id': 'VARCHAR'}) AS s 
            ON st.stop_id = s.stop_id
        JOIN read_csv_auto('../gtfs_static/MBTA_GTFS/routes.txt', types={'route_id': 'VARCHAR'}) AS r
            ON p.route_id = r.route_id
    )
    -- 2. Filtering using QUALIFY to keep only the closest stop for every ping
    
    SELECT 
        vehicle_id,
        route_short_name,
        route_long_name,
        trip_id,
        timestamp,
        stop_name AS closest_stop,
        stop_sequence,
        dist_sq_meters
    FROM stops_for_trip
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY vehicle_id, timestamp 
        ORDER BY dist_sq_meters ASC
    ) = 1
    ORDER BY vehicle_id, timestamp
"""

df_mapped = duckdb.sql(query_map_matching).df()
display(df_mapped.head(20))

,vehicle_id,route_short_name,route_long_name,trip_id,timestamp,closest_stop,stop_sequence,dist_sq_meters
0,R-548B3AFA,NaN,Red Line,76755396,2026-08-14 23:36:28-06:00,Shawmut,160,166.826234
1,R-548B3AFA,NaN,Red Line,76755396,2026-08-14 23:37:16-06:00,Shawmut,160,70657.576366
2,R-548B3AFA,NaN,Red Line,76755396,2026-08-14 23:37:43-06:00,Ashmont,170,225503.810846
3,R-548B3AFA,NaN,Red Line,76755396,2026-08-14 23:37:55-06:00,Ashmont,170,53780.415600
4,y0768,220,Hingham Depot - Quincy Center Station,77117971,2026-08-14 23:36:31-06:00,Station St - Hingham Depot,1,716.409597
5,y0768,220,Hingham Depot - Quincy Center Station,77117971,2026-08-14 23:37:11-06:00,Station St - Hingham Depot,1,361.464770
6,y0768,220,Hingham Depot - Quincy Center Station,77117971,2026-08-14 23:37:27-06:00,Station St - Hingham Depot,1,391.868600
7,y0768,220,Hingham Depot - Quincy Center Station,77117971,2026-08-14 23:37:44-06:00,North St @ Otis St,2,704.842517
8,y0768,220,Hingham Depot - Quincy Center Station,77117971,2026-08-14 23:37:55-06:00,North St @ Otis St,2,451.204263
9,y0768,220,Hingham Depot - Quincy Center Station,77117971,2026-08-14 23:38:12-06:00,North St @ Otis St,2,6816.282021
